# 🖼️ Multi-panel figures

**Time:** ~25 minutes &nbsp;•&nbsp; **Goal:** put several axes in one frame, and make them argue together rather than sit next to each other.

A paper figure is almost never one panel. It is three or four, arranged so that the reader's eye moves from the general to the specific, and the arrangement itself is part of the claim.

You have used `plt.subplots(1, 2)` already. This notebook is the rest of it: grids, shared axes, unequal panels, and the discipline that stops a multi-panel figure becoming a collage.

## Run this first

In [ ]:
# Run this first. Every exercise below uses `df`.
import urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab07/data/"

DEPMAP = "depmap.csv"
if not Path(DEPMAP).exists():
    urllib.request.urlretrieve(DATA_URL + DEPMAP, DEPMAP)

df = pd.read_csv(DEPMAP)

GENES = ["SOX10", "BRAF", "KRAS", "NRAS", "PAX8", "MYB", "CTNNB1", "EGFR",
         "MDM2", "ESR1", "RAN", "RPL5", "RPS6", "PSMA1", "SNRPD1", "EIF4A3",
         "A1BG", "AADAC"]

print(df.shape)
df[["cell_line_name", "oncotree_lineage", "SOX10", "BRAF", "RAN"]].head()

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.axisbelow": True,
    "font.size": 10,
})

## Skill 1 — A grid of axes

`plt.subplots(rows, cols)` returns a `fig` and an **array** of axes.

```python
fig, axes = plt.subplots(2, 3)
axes[0, 1]      # top row, middle column
axes.flat       # all six, in reading order, as a flat sequence
```

With one row or one column the array is 1-D (`axes[1]`); with both it is 2-D (`axes[1, 2]`). `axes.flat` sidesteps the difference entirely, and is what you want whenever you are looping.

In [ ]:
# DEMO - six genes, six panels, one loop
genes = ["RAN", "RPL5", "PSMA1", "SOX10", "MYB", "A1BG"]

fig, axes = plt.subplots(2, 3)
fig.set_size_inches(11, 5)

for ax, gene in zip(axes.flat, genes):
    ax.hist(df[gene], bins=35, color="#5b9df9")
    ax.set_title(gene)

fig.supxlabel("dependency score")
fig.supylabel("cell lines")
fig.tight_layout()
plt.show()

`fig.supxlabel` and `fig.supylabel` put one label on the whole figure instead of six identical ones. With `fig.suptitle` they are the three figure-level text methods worth knowing.

But look carefully at those six panels. Each has its own x-range, so RAN's distribution — which sits around −4 — is drawn in the same physical space as A1BG's, which sits at 0. **The panels look comparable and are not.**

### 🖼️ DIY challenge

1. Redraw the grid with `ax.set_xlim(-5.5, 1)` on every panel. Which of the six genes now looks different from how it looked before?
2. Change the grid to `plt.subplots(3, 2)`. What happens to the panel order?
3. Loop over all 18 genes in `GENES` with `plt.subplots(6, 3)`. You will need a bigger `figsize` and smaller titles — `ax.set_title(gene, fontsize=9)`.

> 💡 `zip(axes.flat, genes)` stops at whichever runs out first. If you have more panels than genes, the extras stay blank — hide them with `ax.set_visible(False)`.

In [ ]:
# TODO - your turn

## Skill 2 — `sharex` and `sharey` are an argument

Setting the same limits by hand on every panel works, and is easy to get wrong. Better to say it once:

```python
fig, axes = plt.subplots(2, 3, sharex=True, sharey=True)
```

Two things happen. The panels are locked to the same range, whatever the data does — and matplotlib **removes the redundant tick labels**, so only the left column and the bottom row are labelled. The figure gets cleaner and more honest at the same time.

A shared axis is a claim: *these panels are on the same scale, so you may compare them by eye.* Do not share axes when that claim is false.

In [ ]:
# DEMO - the same six genes, now genuinely comparable
genes = ["RAN", "RPL5", "PSMA1", "SOX10", "MYB", "A1BG"]

fig, axes = plt.subplots(2, 3, sharex=True, sharey=True)
fig.set_size_inches(11, 5)

for ax, gene in zip(axes.flat, genes):
    ax.hist(df[gene], bins=35, color="#5b9df9")
    ax.axvline(0, color="#9ca3af", linewidth=0.8)
    ax.set_title(gene)

fig.supxlabel("dependency score")
fig.supylabel("cell lines")
fig.suptitle("Three common essentials, two selective genes, one nobody needs")
fig.tight_layout()
plt.show()

Now the figure makes its own point. The top row sits far left and the bottom row is jammed against zero, and you can see that from across a room — which is the test a conference poster has to pass.

### 🖼️ DIY challenge

1. Rebuild it with `sharey=False`. Is the figure more readable or less honest? Both?
2. Make a 1×3 boxplot figure — SOX10, MYB and PAX8, each across the same four lineages — with `sharey=True`. Does sharing help here?
3. Try `sharex="col"` on a 2×2 grid. What does the string form do that `True` does not?

> 💡 `sharex` takes `True`, `"col"`, `"row"` or `False`. `"col"` shares down each column only — right when your columns are different measurements.

In [ ]:
# TODO - your turn

## Skill 3 — Unequal panels with `subplot_mosaic`

A real figure is rarely a neat grid: one big panel and two small ones, or a wide strip along the bottom. `plt.subplot_mosaic` lets you draw the layout as text and get back a **dictionary** of axes keyed by the letters you used.

```python
fig, axd = plt.subplot_mosaic('''
    AAB
    AAC
''')
axd["A"]    # the big square on the left
axd["B"]    # top right
```

Each letter becomes one axes, occupying every cell it appears in. `.` leaves a cell empty. It is far easier to read than the `gridspec` calls it replaces, and you can restructure a figure by editing the picture.

In [ ]:
# DEMO - one big panel, two supporting ones
fig, axd = plt.subplot_mosaic('''
    AAB
    AAC
''')
fig.set_size_inches(11, 5)

skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]

axd["A"].scatter(rest["BRAF"], rest["SOX10"], s=10, alpha=0.3,
                 color="#9ca3af", label="other lineages")
axd["A"].scatter(skin["BRAF"], skin["SOX10"], s=16, alpha=0.9,
                 color="#5b9df9", label="skin")
axd["A"].set_xlabel("BRAF dependency")
axd["A"].set_ylabel("SOX10 dependency")
axd["A"].legend()

axd["B"].hist(df["SOX10"], bins=40, color="#5b9df9")
axd["B"].set_title("SOX10, all lines", fontsize=10)

axd["C"].hist(skin["SOX10"], bins=20, color="#5b9df9")
axd["C"].set_title("SOX10, skin only", fontsize=10)
axd["C"].set_xlabel("dependency")

fig.tight_layout()
plt.show()

The two small panels on the right are the same measurement on different subsets, so they should be on the same x-scale — and they are not. Fixing that is the first DIY question, and it is the mistake this layout invites.

### 🖼️ DIY challenge

1. Put `axd["B"]` and `axd["C"]` on the same x-range. Does the story change?
2. Redesign the layout: two panels on top, one wide one underneath. You only have to edit the string.
3. Add a fourth panel `D` and put the lineage bar chart in it.
4. `subplot_mosaic` takes `sharex=True` too. Try it — what does it do to a layout where the panels are different sizes?

> 💡 The string is whitespace-sensitive in the sense that every row must be the same length. Line them up.

In [ ]:
# TODO - your turn

## Skill 4 — 🏆 One figure that tells the whole story

You have a finding: **SOX10 is a selective dependency of melanoma, and it travels with BRAF.**

Making a reader believe it takes three things, in this order:

1. **the context** — SOX10 across all 1,165 lines, so they see the tail exists
2. **the split** — SOX10 by lineage, so they see the tail is one tissue
3. **the mechanism-shaped detail** — SOX10 against BRAF, skin highlighted, so they see the two dependencies coincide

That is a three-panel figure, and the order is the argument. Panels are read left to right; a reader who meets the scatter first has no reason to care about it.

In [ ]:
# DEMO - the whole story, assembled
fig, axd = plt.subplot_mosaic('''
    ABC
''')
fig.set_size_inches(13, 3.8)

BLUE, GREY = "#5b9df9", "#9ca3af"

# A - context
axd["A"].hist(df["SOX10"], bins=40, color=BLUE)
axd["A"].axvline(0, color="#374151", linewidth=0.8)
axd["A"].set_xlabel("SOX10 dependency")
axd["A"].set_ylabel("cell lines")
axd["A"].set_title(f"A  All lineages (n = {len(df):,})", loc="left", fontsize=11)

# B - the split
names = ["Skin", "Lung", "Lymphoid", "Breast", "CNS/Brain"]
groups = [df[df["oncotree_lineage"] == n]["SOX10"] for n in names]
bp = axd["B"].boxplot(groups, tick_labels=names, patch_artist=True)
for patch, name in zip(bp["boxes"], names):
    patch.set_facecolor(BLUE if name == "Skin" else GREY)
axd["B"].axhline(0, color="#374151", linewidth=0.8)
axd["B"].set_ylabel("SOX10 dependency")
axd["B"].tick_params(axis="x", labelsize=9, rotation=30)
axd["B"].set_title("B  Split by lineage", loc="left", fontsize=11)

# C - the detail
skin = df[df["oncotree_lineage"] == "Skin"]
rest = df[df["oncotree_lineage"] != "Skin"]
axd["C"].scatter(rest["BRAF"], rest["SOX10"], s=10, alpha=0.3, color=GREY,
                 label=f"other (r = {rest['BRAF'].corr(rest['SOX10']):.2f})")
axd["C"].scatter(skin["BRAF"], skin["SOX10"], s=16, alpha=0.9, color=BLUE,
                 label=f"skin (r = {skin['BRAF'].corr(skin['SOX10']):.2f})")
axd["C"].set_xlabel("BRAF dependency")
axd["C"].set_ylabel("SOX10 dependency")
axd["C"].legend(fontsize=8, loc="lower right")
axd["C"].set_title("C  SOX10 travels with BRAF", loc="left", fontsize=11)

fig.tight_layout()
fig.savefig("sox10_story.svg", bbox_inches="tight")
fig.savefig("sox10_story.png", dpi=300, bbox_inches="tight")
plt.show()

Three things in there are worth stealing.

`patch_artist=True` makes the boxes fillable, and `bp["boxes"]` hands you the patch objects so you can colour one group differently from the rest. Highlighting one category and greying the others is the most reliable way to point at something without an arrow.

`loc="left"` on a panel title puts the letter where a journal wants it. And the correlations are computed **into the legend labels** with an f-string, so the number and the colour it describes cannot drift apart.

### 🏆 DIY finale — your own three-panel figure

Pick a different gene with a lineage story — `MYB` (Lymphoid and Myeloid), `PAX8` (Ovary/Fallopian Tube), or `EGFR` (Head and Neck) — and build the same three-panel argument for it.

Requirements:

1. panel A: the distribution across all lines, with a zero line
2. panel B: a boxplot across five lineages, the relevant one highlighted
3. panel C: a scatter against a second gene of your choosing, with `r` in the legend
4. panel letters, `loc="left"`, and `n` stated somewhere
5. one colour scheme across all three panels, defined once at the top as a variable
6. saved as both `.svg` and `.png` at `dpi=300`

Then write three sentences: what the figure claims, what the weakest panel is, and one thing a sceptical reader would ask for next.

> 💡 One of these genes has a much weaker story than the others. Finding that out and saying so is a better outcome than a tidy figure of a result that is not there.

In [ ]:
# TODO - your turn

## 🔭 Explore further

- Arranging multiple axes, the official guide: <https://matplotlib.org/stable/users/explain/axes/arranging_axes.html>
- `subplot_mosaic` in depth, with layouts you would not have thought of: <https://matplotlib.org/stable/gallery/subplots_axes_and_figures/mosaic.html>
- **Extra notebook:** `extras/01_scales_and_seaborn` — log axes, twin axes, and what seaborn has been doing for you all along.

## Recap

You can now:

- build a grid of axes and loop over it with `axes.flat`
- use `sharex`/`sharey` to make panels genuinely comparable, and know that sharing is a **claim**
- lay out unequal panels by drawing them, with `subplot_mosaic`
- highlight one group with `patch_artist` and a colour, rather than an arrow
- label panels the way a journal expects, and compute numbers into the labels
- assemble context → split → detail into a figure that makes an argument

That is the matplotlib toolkit. Everything else you meet — seaborn, plotnine, pandas' own `.plot()` — is drawing on an `ax` you now know how to control.